In [1]:
# Step 1: Fix version issues
!pip uninstall -y elasticsearch
!pip install "elasticsearch>=8.0.0,<9.0.0"

Found existing installation: elasticsearch 9.0.1
Uninstalling elasticsearch-9.0.1:
  Successfully uninstalled elasticsearch-9.0.1
   ---------------------------------------- 0.0/906.3 kB ? eta -:--:--
   --- ------------------------------------ 81.9/906.3 kB 4.5 MB/s eta 0:00:01
   --- ------------------------------------ 81.9/906.3 kB 4.5 MB/s eta 0:00:01
   ---------- ----------------------------- 245.8/906.3 kB 2.1 MB/s eta 0:00:01
   ---------- ----------------------------- 245.8/906.3 kB 2.1 MB/s eta 0:00:01
   ---------- ----------------------------- 245.8/906.3 kB 2.1 MB/s eta 0:00:01
   ---------- ----------------------------- 245.8/906.3 kB 2.1 MB/s eta 0:00:01
   -------------- ------------------------- 327.7/906.3 kB 1.0 MB/s eta 0:00:01
   -------------- ------------------------- 327.7/906.3 kB 1.0 MB/s eta 0:00:01
   -------------- ------------------------- 327.7/906.3 kB 1.0 MB/s eta 0:00:01
   ----------------- -------------------- 409.6/906.3 kB 946.4 kB/s eta 0:00:01
 

In [2]:
import pandas as pd

In [3]:
import sqlite3

In [4]:
df = pd.read_csv("data.csv")

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 936 entries, 0 to 935
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   timestamp  936 non-null    object
 1   title      936 non-null    object
 2   summary    888 non-null    object
 3   link       936 non-null    object
 4   source     936 non-null    object
dtypes: object(5)
memory usage: 36.7+ KB


In [6]:
df.head(5)

,timestamp,title,summary,link,source
0,2025-05-22T11:29:05.503214,LIVETwo Israeli embassy staff killed and suspe...,NaN,https://www.bbc.comhttps://www.bbc.com/news/li...,bbc
1,2025-05-22T11:29:05.503214,US Jewish museum shooting suspect was mistaken...,NaN,https://www.bbc.com/news/articles/cz63g3441wgo,bbc
2,2025-05-22T11:29:05.503214,LIVE'Multiple fatalities' on private plane tha...,NaN,https://www.bbc.comhttps://www.bbc.com/news/li...,bbc
3,2025-05-22T11:29:05.503214,LIVEUS House passes Trump tax and spending meg...,NaN,https://www.bbc.comhttps://www.bbc.com/news/li...,bbc
4,2025-05-22T11:29:05.503214,"Watch: Deep inside a Norwegian mountain, Nato ...",NaN,https://www.bbc.com/news/videos/cy0jjxnd8weo,bbc


## Store in SQL

In [7]:
conn = sqlite3.connect("news.db")
df.to_sql("news_articles", conn, if_exists='replace', index=False)

936

## Store in ElasticSearch

In [8]:
!pip install elasticsearch

In [9]:
from elasticsearch import Elasticsearch

In [19]:
# Step 1: Convert 'timestamp' to datetime safely (invalid ones become NaT)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Step 2: Replace NaT with None manually
df['timestamp'] = df['timestamp'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Step 3: Replace all other NaNs with None
df = df.where(pd.notnull(df), None)


In [20]:
es = Elasticsearch("http://localhost:9200")

# Test connection
print(es.info())

{'name': 'd17b0c6368b7', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'uy2QIg5CS0OJ81JoX1ohIQ', 'version': {'number': '8.13.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '09df99393193b2c53d92899662a8b8b3c55b45cd', 'build_date': '2024-03-22T03:35:46.757803203Z', 'build_snapshot': False, 'lucene_version': '9.10.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [21]:
# Define the index name
index_name = "news-index"

# Loop through each row and upload it to ElasticSearch
for i, row in df.iterrows():
    doc = row.to_dict()  # convert row to dictionary
    es.index(index=index_name, document=doc)

print(f"{len(df)} documents indexed into ElasticSearch.")

936 documents indexed into ElasticSearch.


In [22]:
df_sql = pd.read_sql("SELECT * FROM news_articles", conn)

# Show first 5 rows
df_sql.head()

,timestamp,title,summary,link,source
0,2025-05-22T11:29:05.503214,LIVETwo Israeli embassy staff killed and suspe...,None,https://www.bbc.comhttps://www.bbc.com/news/li...,bbc
1,2025-05-22T11:29:05.503214,US Jewish museum shooting suspect was mistaken...,None,https://www.bbc.com/news/articles/cz63g3441wgo,bbc
2,2025-05-22T11:29:05.503214,LIVE'Multiple fatalities' on private plane tha...,None,https://www.bbc.comhttps://www.bbc.com/news/li...,bbc
3,2025-05-22T11:29:05.503214,LIVEUS House passes Trump tax and spending meg...,None,https://www.bbc.comhttps://www.bbc.com/news/li...,bbc
4,2025-05-22T11:29:05.503214,"Watch: Deep inside a Norwegian mountain, Nato ...",None,https://www.bbc.com/news/videos/cy0jjxnd8weo,bbc
